# JSON Metadata Generator for ADLS

This notebook generates a JSON metadata file with a parameterized structure and writes it to an Azure Data Lake Storage (ADLS) location.

## Key Features
- **Fully Parameterized**: All JSON attributes are controlled via Databricks widgets
- **ADF-Ready**: Designed to be called from Azure Data Factory pipelines
- **Dynamic Values**: `dataProducerName` and `dataObjectTechnicalName` update based on parameters
- **ADLS Integration**: Direct write to ADLS using `abfss://` protocol

## Widget Parameters

| Parameter | Description | Example Value |
|-----------|-------------|---------------|
| `dataProducerName` | Name of the data producer | `rmm-cas-calculators` |
| `dataObjectTechnicalName` | Technical name of the data object | `rmm_v2_ut1_obv_config_df` |
| `dataObjectLakeLayer` | Lake layer (raw/defined/curated) | `defined` |
| `dataObjectTechnicalIdSourceObject` | Source object IDs as JSON array | `[]` or `["obj1", "obj2"]` |
| `dataObjectProcessingJobPlannedFrequencyType` | Processing frequency | `Day` / `Hour` / `Week` |
| `dataObjectDeliveryLocation` | ADLS delivery path (https:// format) | `https://storage.dfs.core.windows.net/...` |
| `dataObjectVersion` | Version number | `1` |
| `dataObjectType` | Type of data object | `periodic_snapshot` |
| `jobId` | Job ID (use "null" for null value) | `null` or `12345` |
| `dataObjectProcessingJobNumberOfLinesWritten` | Number of lines written | `0` or `1000` |
| `dataObjectProcessingExtractionType` | Extraction type | `Snapshot` / `Incremental` |
| `dataObjectEDLLoadDatetime` | Load datetime in ISO format | `2026-04-02T00:00:00Z` |
| `outputJsonPath` | Output path for JSON file (abfss:// format) | `abfss://container@storage.dfs.core.windows.net/path/file.json` |

## Usage from Azure Data Factory

### Notebook Activity Configuration

```json
{
  "notebookPath": "/Users/net.rajeev@gmail.com/Databricks_Learning/Json_file",
  "baseParameters": {
    "dataProducerName": "@{pipeline().parameters.producerName}",
    "dataObjectTechnicalName": "@{pipeline().parameters.technicalName}",
    "dataObjectDeliveryLocation": "@{pipeline().parameters.deliveryLocation}",
    "dataObjectEDLLoadDatetime": "@{utcnow()}",
    "outputJsonPath": "@{pipeline().parameters.outputPath}"
  }
}
```

### Example ADF Pipeline Parameter Mapping

1. Add a **Databricks Notebook Activity** to your ADF pipeline
2. Set the **Notebook path** to this notebook's path
3. In **Base parameters**, map ADF pipeline parameters to notebook widgets
4. The notebook will execute and write the JSON to the specified ADLS location

## Output Format

The generated JSON follows this structure:

```json
{
  "dataProducerName": "<value>",
  "dataObjectTechnicalName": "<value>",
  "dataObjectLakeLayer": "<value>",
  "dataObjectTechnicalIdSourceObject": [],
  "dataObjectProcessingJobPlannedFrequencyType": "<value>",
  "dataObjectDeliveryLocation": "<value>",
  "dataObjectVersion": 1,
  "dataObjectType": "<value>",
  "operationalMetadata": {
    "jobId": null,
    "dataObjectProcessingJobNumberOfLinesWritten": "<value>",
    "dataObjectProcessingExtractionType": "<value>",
    "dataObjectEDLLoadDatetime": "<value>"
  }
}
```

In [0]:
import json
from datetime import datetime

# Remove existing widgets (clean slate for re-runs)
dbutils.widgets.removeAll()

# Create widgets for all JSON attributes
dbutils.widgets.text("dataProducerName", "rmm-cas-calculators", "Data Producer Name")
dbutils.widgets.text("dataObjectTechnicalName", "rmm_v2_ut1_obv_config_df", "Data Object Technical Name")
dbutils.widgets.text("dataObjectLakeLayer", "defined", "Data Object Lake Layer")
dbutils.widgets.text("dataObjectTechnicalIdSourceObject", "[]", "Source Object IDs (JSON array)")
dbutils.widgets.text("dataObjectProcessingJobPlannedFrequencyType", "Day", "Processing Frequency Type")
dbutils.widgets.text("dataObjectDeliveryLocation", "https://edlcorestdeudev0001.dfs.core.windows.net/rmm-cas-calculators/rmm_v2_ut1_obv_config_df/1/data/is_prod_run=True/TIMESTAMP=20260402T000000Z/", "Delivery Location (ADLS Path)")
dbutils.widgets.text("dataObjectVersion", "1", "Data Object Version")
dbutils.widgets.text("dataObjectType", "periodic_snapshot", "Data Object Type")
dbutils.widgets.text("jobId", "null", "Job ID")
dbutils.widgets.text("dataObjectProcessingJobNumberOfLinesWritten", "0", "Number of Lines Written")
dbutils.widgets.text("dataObjectProcessingExtractionType", "Snapshot", "Extraction Type")
dbutils.widgets.text("dataObjectEDLLoadDatetime", "2026-04-02T00:00:00Z", "EDL Load Datetime (ISO format)")
dbutils.widgets.text("outputJsonPath", "abfss://rmm-cas-calculators@edlcorestdeudev0001.dfs.core.windows.net/metadata/data_object_metadata.json", "Output JSON File Path (ADLS)")

print("✓ All widgets created successfully")
print("\nCurrent parameter values:")
print(f"  dataProducerName: {dbutils.widgets.get('dataProducerName')}")
print(f"  dataObjectTechnicalName: {dbutils.widgets.get('dataObjectTechnicalName')}")

In [0]:
def generate_json_metadata():
    """
    UDF to generate JSON metadata file from widget parameters.
    Returns the constructed JSON object.
    """
    # Read all widget values
    data_producer_name = dbutils.widgets.get("dataProducerName")
    data_object_technical_name = dbutils.widgets.get("dataObjectTechnicalName")
    data_object_lake_layer = dbutils.widgets.get("dataObjectLakeLayer")
    source_objects_str = dbutils.widgets.get("dataObjectTechnicalIdSourceObject")
    frequency_type = dbutils.widgets.get("dataObjectProcessingJobPlannedFrequencyType")
    delivery_location = dbutils.widgets.get("dataObjectDeliveryLocation")
    version = dbutils.widgets.get("dataObjectVersion")
    object_type = dbutils.widgets.get("dataObjectType")
    job_id = dbutils.widgets.get("jobId")
    lines_written = dbutils.widgets.get("dataObjectProcessingJobNumberOfLinesWritten")
    extraction_type = dbutils.widgets.get("dataObjectProcessingExtractionType")
    edl_load_datetime = dbutils.widgets.get("dataObjectEDLLoadDatetime")
    
    # Parse source objects (handle JSON array or empty array)
    try:
        source_objects = json.loads(source_objects_str) if source_objects_str.strip() else []
    except json.JSONDecodeError:
        source_objects = []
    
    # Parse jobId (convert "null" string to actual null)
    job_id_value = None if job_id.lower() == "null" else job_id
    
    # Construct the JSON structure
    metadata = {
        "dataProducerName": data_producer_name,
        "dataObjectTechnicalName": data_object_technical_name,
        "dataObjectLakeLayer": data_object_lake_layer,
        "dataObjectTechnicalIdSourceObject": source_objects,
        "dataObjectProcessingJobPlannedFrequencyType": frequency_type,
        "dataObjectDeliveryLocation": delivery_location,
        "dataObjectVersion": int(version),
        "dataObjectType": object_type,
        "operationalMetadata": {
            "jobId": job_id_value,
            "dataObjectProcessingJobNumberOfLinesWritten": lines_written,
            "dataObjectProcessingExtractionType": extraction_type,
            "dataObjectEDLLoadDatetime": edl_load_datetime
        }
    }
    
    return metadata


def write_json_to_adls(json_data, output_path):
    """
    Write JSON data to ADLS location.
    
    Args:
        json_data: Dictionary to write as JSON
        output_path: ADLS path (abfss:// format)
    """
    # Convert to formatted JSON string
    json_string = json.dumps(json_data, indent=2)
    
    # Write to ADLS
    dbutils.fs.put(output_path, json_string, overwrite=True)
    
    return output_path


# Generate the JSON metadata
metadata_json = generate_json_metadata()

# Display the generated JSON
print("✓ JSON metadata generated successfully")
print("\n" + "="*60)
print("Generated JSON:")
print("="*60)
print(json.dumps(metadata_json, indent=2))
print("="*60)

# Try to write to ADLS (requires storage credentials to be configured)
try:
    output_path = dbutils.widgets.get("outputJsonPath")
    written_path = write_json_to_adls(metadata_json, output_path)
    print(f"\n✓ JSON file written to: {written_path}")
except Exception as e:
    print(f"\n⚠ Could not write to ADLS (credentials not configured): {str(e)[:100]}...")
    print("\nTo write to ADLS from ADF:")
    print("  1. Ensure the Databricks workspace has access to the storage account")
    print("  2. Configure storage credentials in Databricks or use Azure Key Vault")
    print("  3. The notebook will write successfully when called from ADF with proper permissions")